# Extended Weather Ingestion Job for Multi-Hour Forecasting

This notebook ingests an extended hourly weather range used by the **multi-hour forecasting branch** of the project.

It downloads hourly weather data for the Toronto area covering:

- recent past days
- the current day
- the next 7 forecast days

The purpose is to provide a consistent meteorological input source for the weekly forecasting engine, which requires both:
- recent weather context
- future hourly weather conditions

This notebook represents the **weather ingestion layer for the 1-week forecasting workflow**.

## Process Overview

This notebook performs the following steps:

### 1. Define the extended weather range
The job computes a date window covering:
- `today - past_days_back`
- `today + future_days_ahead`

This ensures the dataset contains both recent and forward-looking hourly weather records.

### 2. Query the Open-Meteo API
The notebook requests hourly weather data for downtown Toronto, including:
- temperature
- apparent temperature

### 3. Save the raw JSON payload
The full weather response is stored in DBFS for:
- traceability
- reproducibility
- debugging

### 4. Build structured hourly weather records
The JSON response is converted into a structured table containing:
- year, month, day, hour
- local weather timestamp
- temperature fields
- ingestion metadata
- requested date range parameters

### 5. Write to the bronze weather range table
The processed records are appended into:
- `workspace.default.bronze_weather_hourly_minimal_range`

### 6. Refresh analytical views
Two views are created:

- `vw_weather_hourly_minimal_range_latest`
  - latest available weather record per hour

- `vw_weather_hourly_minimal_range_future`
  - latest records restricted to future timestamps only

These views are used directly by downstream forecasting jobs.

In [0]:
# Databricks notebook source
# ============================================================
# WEATHER INGESTION JOB - DEFINITIVE VERSION
# Serverless + Unity Catalog safe
#
# Coverage:
#   past_days_back     = 2
#   future_days_ahead  = 7
#
# Downloads hourly weather for:
#   [today - 2 days, today + 7 days]
#
# Variables:
#   - temperature_2m
#   - apparent_temperature
#
# Output:
#   - workspace.default.bronze_weather_hourly_minimal_range
#   - workspace.default.vw_weather_hourly_minimal_range_latest
#   - workspace.default.vw_weather_hourly_minimal_range_future
# ============================================================

from pyspark.sql import functions as F
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

# ------------------------------------------------------------
# 0) CONFIG
# ------------------------------------------------------------
UC_CATALOG = "workspace"
UC_SCHEMA  = "default"

TBL_WEATHER = f"{UC_CATALOG}.{UC_SCHEMA}.bronze_weather_hourly_minimal_range"

VW_LATEST = f"{UC_CATALOG}.{UC_SCHEMA}.vw_weather_hourly_minimal_range_latest"
VW_FUTURE = f"{UC_CATALOG}.{UC_SCHEMA}.vw_weather_hourly_minimal_range_future"

TIMEZONE = "America/Toronto"
LATITUDE = 43.65
LONGITUDE = -79.38

PAST_DAYS_BACK = 3
FUTURE_DAYS_AHEAD = 7

RAW_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/raw/weather"

# ------------------------------------------------------------
# 1) DATE RANGE
# ------------------------------------------------------------
now_local = datetime.now(ZoneInfo(TIMEZONE))
today_local = now_local.date()

start_date = (today_local - timedelta(days=PAST_DAYS_BACK)).strftime("%Y-%m-%d")
end_date   = (today_local + timedelta(days=FUTURE_DAYS_AHEAD)).strftime("%Y-%m-%d")

# ------------------------------------------------------------
# 2) HELPERS
# ------------------------------------------------------------
def fetch_weather():
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "hourly": "temperature_2m,apparent_temperature",
        "timezone": TIMEZONE,
        "start_date": start_date,
        "end_date": end_date
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

def write_raw_json(payload: dict):
    yyyy = now_local.strftime("%Y")
    mm   = now_local.strftime("%m")
    dd   = now_local.strftime("%d")
    ts   = now_local.strftime("%Y%m%dT%H%M%S")

    out_dir = f"{RAW_DIR}/forecast_range/year={yyyy}/month={mm}/day={dd}"
    out_file = f"{out_dir}/forecast_range_{ts}.json"

    dbutils.fs.put(out_file, json.dumps(payload, ensure_ascii=False), overwrite=True)
    print(f"RAW saved: {out_file}")

def build_records(data: dict):
    hourly = data.get("hourly", {})
    times = hourly.get("time", [])
    temps = hourly.get("temperature_2m", [])
    app_temps = hourly.get("apparent_temperature", [])

    records = []
    for i in range(len(times)):
        dt = datetime.fromisoformat(times[i])

        records.append({
            "range_start_date": start_date,
            "range_end_date": end_date,
            "past_days_back": PAST_DAYS_BACK,
            "future_days_ahead": FUTURE_DAYS_AHEAD,
            "year": dt.year,
            "month": dt.month,
            "day": dt.day,
            "hour": dt.hour,
            "weather_ts_local": times[i],
            "temperature_2m_c": temps[i] if i < len(temps) else None,
            "apparent_temperature_c": app_temps[i] if i < len(app_temps) else None
        })
    return records

# ------------------------------------------------------------
# 3) INGEST
# ------------------------------------------------------------
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{UC_SCHEMA}")

print("========================================")
print("Weather definitive ingestion started")
print(f"Target table         : {TBL_WEATHER}")
print(f"Timezone             : {TIMEZONE}")
print(f"Start date           : {start_date}")
print(f"End date             : {end_date}")
print(f"Past days back       : {PAST_DAYS_BACK}")
print(f"Future days ahead    : {FUTURE_DAYS_AHEAD}")
print("========================================")

data = fetch_weather()
write_raw_json(data)

records = build_records(data)
pdf = pd.DataFrame(records)
pdf = pdf.where(pd.notnull(pdf), None)

df = spark.createDataFrame(pdf)

df = (
    df.withColumn("weather_ts_local", F.to_timestamp("weather_ts_local"))
      .withColumn("weather_date_local", F.to_date("weather_ts_local"))
      .withColumn("ingested_at_utc", F.current_timestamp())
)

(
    df.write
      .format("delta")
      .mode("append")
      .saveAsTable(TBL_WEATHER)
)

print(f"Weather saved: {TBL_WEATHER}")

# ------------------------------------------------------------
# 4) VIEW: LATEST PER HOUR
# ------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {VW_LATEST} AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY year, month, day, hour
               ORDER BY ingested_at_utc DESC
           ) AS rn
    FROM {TBL_WEATHER}
)
SELECT *
FROM ranked
WHERE rn = 1
""")

print(f"Latest weather view refreshed: {VW_LATEST}")

# ------------------------------------------------------------
# 5) VIEW: FUTURE HOURS ONLY
# ------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE VIEW {VW_FUTURE} AS
WITH ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY year, month, day, hour
               ORDER BY ingested_at_utc DESC
           ) AS rn
    FROM {TBL_WEATHER}
),
latest_only AS (
    SELECT *
    FROM ranked
    WHERE rn = 1
)
SELECT *
FROM latest_only
WHERE weather_ts_local >= from_utc_timestamp(current_timestamp(), '{TIMEZONE}')
""")

print(f"Future-only weather view refreshed: {VW_FUTURE}")

# ------------------------------------------------------------
# 6) VALIDATION
# ------------------------------------------------------------
print("Coverage validation:")
display(
    spark.sql(f"""
    SELECT
        MIN(weather_ts_local) AS min_ts,
        MAX(weather_ts_local) AS max_ts,
        COUNT(*) AS rows_cnt,
        MIN(temperature_2m_c) AS min_temp,
        MAX(temperature_2m_c) AS max_temp
    FROM {VW_LATEST}
    """)
)

print("Future-only validation:")
display(
    spark.sql(f"""
    SELECT
        MIN(weather_ts_local) AS min_ts,
        MAX(weather_ts_local) AS max_ts,
        COUNT(*) AS rows_cnt
    FROM {VW_FUTURE}
    """)
)

print("Sample rows:")
display(
    spark.sql(f"""
    SELECT
        year, month, day, hour,
        weather_ts_local,
        temperature_2m_c,
        apparent_temperature_c
    FROM {VW_LATEST}
    ORDER BY weather_ts_local
    LIMIT 100
    """)
)

print("========================================")
print("Weather definitive ingestion finished")
print("========================================")

## Outputs

This notebook produces the following outputs:

### 1. Raw JSON weather files
The complete extended weather payload is stored in DBFS for auditability and future reprocessing.

---

### 2. Bronze weather range table
Structured hourly weather records are written into:

- `workspace.default.bronze_weather_hourly_minimal_range`

This table stores:
- hourly local timestamps
- temperature
- apparent temperature
- requested date window metadata
- ingestion timestamp

---

### 3. Latest weather range view
The notebook refreshes:

- `workspace.default.vw_weather_hourly_minimal_range_latest`

This view returns the latest weather record for each `(year, month, day, hour)` combination.

---

### 4. Future-only weather view
The notebook also refreshes:

- `workspace.default.vw_weather_hourly_minimal_range_future`

This view contains only future hourly records and is specifically designed for the multi-hour forecasting engine.

## Key Insights and Summary

### 1. The weekly forecasting branch requires future weather coverage
Unlike the 1-hour prediction pipeline, the multi-hour forecasting engine needs weather values for many future target hours. This notebook provides that extended forecast horizon.

---

### 2. Recent past weather is also preserved
By including several past days, the notebook maintains temporal continuity and supports validation, debugging, and consistency checks across recent hourly records.

---

### 3. The design separates current-hour and future-hour weather use cases
This notebook exposes two complementary views:

- a latest-per-hour view for complete hourly coverage
- a future-only view tailored to recursive forecasting

This separation simplifies downstream consumption and reduces unnecessary filtering logic in forecasting jobs.

---

### 4. The notebook provides a lightweight but operationally useful weather layer
Although it only ingests a minimal set of weather variables, these variables are highly relevant to bike demand forecasting and are sufficient for the current modeling approach.

---

### 5. Business relevance
Weather conditions can strongly influence bike-sharing usage patterns, especially when forecasting demand over a 7-day horizon.

This notebook ensures the weekly forecasting pipeline has access to a consistent and ready-to-use weather source, improving prediction realism for operational planning and future station risk estimation.

In practical terms, this notebook is the **weather forecast provider for the multi-hour / 1-week prediction branch**.